# Week 9: Sampling, estimation, inference

DS5110 Essentials of Data Science. In-class notebook for both Week 9 sessions.

Dataset: Pima Indians Diabetes (Smith et al., 1988), as left at the end of Week 8:
sentinel zeros decoded to `NaN`, nothing imputed.

| Session | Sections |
| --- | --- |
| A | §1 Selection bias, §2 Survivorship bias, §3 Sampling distributions |
| B | §4 Bootstrap and confidence intervals, §5 Permutation tests and p-values, §6 Multiple comparisons |

Week 8 asked how a value went missing. Week 9 asks how a row got into the file, and how far chance can move a number computed from those rows.

Note: every section starts its own random generator. A section gives the same numbers however often the cells above it ran.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
        "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
RAW = Path("data/raw/pima.csv")

if not RAW.exists():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    pd.read_csv(URL, header=None, names=COLS).to_csv(RAW, index=False)

raw = pd.read_csv(RAW)

# Carried over from Week 8 §2: zero means "not measured" in these five columns only
ZERO_MEANS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df = raw.copy()
df[ZERO_MEANS_MISSING] = df[ZERO_MEANS_MISSING].replace(0, np.nan)
FEATURES = [c for c in COLS if c != "Outcome"]

def zscore(s):
    return (s - s.mean()) / s.std()

def logistic(x):
    return 1 / (1 + np.exp(-x))

df.shape, int(df.isna().sum().sum())

((768, 9), 652)

Expected: `((768, 9), 652)`. Same table, same 652 missing cells as Week 8.

## §1 Selection bias

Problem: Week 8 ended with `df.dropna()` keeping 392 of 768 patients. Every number computed on those rows describes the 392.

Idea: name the target population and the selection rule before computing anything.

- Target population: the people the conclusion is about.
- Frame: the list a sample can be drawn from.
- Sample: the rows that reached the file.
- Probability sample: every unit has a known, nonzero chance of selection. Convenience sample: whoever was easy to reach.

Note: the 768 are themselves a selected group. Women, 21 or older, of Pima heritage, in a long-running study near Phoenix, Arizona.
A mean computed here says nothing direct about other populations.

In [ ]:
# Exercise 1.1
# Task: write summary(d), returning a Series with n, outcome_rate, glucose (mean) and age (mean).
#   Build a 2-row table: all rows of df, and the complete rows df.dropna().
# Expected output: n 768 and 392. Outcome rate about 0.35 and 0.33.
#   Mean age lower in the complete rows.
# YOUR CODE HERE

Question: complete-case analysis is a sample. What rule selected it?

Next: a simulation where the truth is known. Pretend the 763 patients with a Glucose value are the whole population.
A volunteer study recruits from them, and older patients volunteer more often.

In [ ]:
pop = df.dropna(subset=["Glucose"]).reset_index(drop=True)
mu = pop["Glucose"].mean()
rng = np.random.default_rng(5110)

p_volunteer = logistic(1.5 * zscore(pop["Age"]) - 1.5)     # older → more likely to join
volunteers = pop[rng.random(len(pop)) < p_volunteer]
print(len(volunteers), "volunteers")
print(round(mu, 1), "population mean glucose")
print(round(volunteers["Glucose"].mean() - mu, 1), "bias of the volunteer mean")

Expected: roughly a fifth of the 763 volunteer. Bias positive: glucose rises with age, and older patients are over-represented.

Idea: reweight. A volunteer from an under-represented age group counts for more.
Weight = population share of the group / sample share of the group (post-stratification).

Note: this needs the population shares from outside the sample: a census, a registry, the study roster.

In [ ]:
# Exercise 1.2
# Task: write reweighted_bias(sample). Steps:
#   cut Age into BINS for pop and for the sample (pd.cut),
#   compute each group's share in pop and in the sample (value_counts(normalize=True)),
#   weight each sampled row by pop share / sample share of its group,
#   return the weighted mean Glucose (np.average with weights=) minus mu.
#   Apply it to volunteers.
# Expected output: a bias clearly smaller than the unweighted one, but not 0.
# YOUR CODE HERE

In [ ]:
# Exercise 1.3
# Task: a second study recruits patients who report symptoms, so higher glucose → more likely to join.
#   Draw this sample with p = logistic(1.5 * zscore(pop["Glucose"]) - 1.5).
#   Report the unweighted bias and reweighted_bias(sample).
# Expected output: both biases large and positive. Reweighting by age barely helps.
# YOUR CODE HERE

Question: why does age reweighting rescue the volunteer study but not the symptom study?

Note: the volunteer bias does not reach 0. Within each age bin, volunteers are still older than non-volunteers.
Coarse groups → partial correction.

In other words: selection on rows behaves like missingness in cells (Week 8 §4).
Selection on an observed column → reweight on it. Selection on the outcome itself → nothing in the data fixes it.

Takeaway: name the population and the selection rule before the first mean.

## §2 Survivorship bias

Problem: a clinic dashboard tracks mean glucose of its active panel. Each year the number falls. Is the panel getting healthier?

Idea: ask which rows were removed before the data arrived, and by what rule.

- Survivorship bias: analysing only the units that passed a filter, when the filter is related to the outcome.
- Bombers returning with holes in the wings: armour belongs where the returning planes had none (Wald, 1943).
- Fund databases that drop closed funds. Satisfaction surveys of current customers. Default models trained on approved loans only.

Note: the removed rows leave no trace in the file. Week 8's `isna()` cannot count them.

Simulation: the 763 patients of §1 start in the panel. Each year, patients with glucose of 140 or more are referred
to a specialist program with probability 0.30. Everyone else leaves with probability 0.03. No patient's glucose ever changes.

In [ ]:
# Exercise 2.1
# Task: start with panel = pop. For year 0 to 5, record year, n and mean Glucose of the panel,
#   then remove patients: each leaves with probability p_leave
#   (np.where(panel["Glucose"] >= 140, 0.30, 0.03)), using rng.random.
#   Show the six records as a DataFrame.
# Expected output: n falls from 763. Mean glucose falls every year, from about 121.7.
# YOUR CODE HERE

Note: `panel` keeps the original row labels of `pop`. `pop.drop(panel.index)` gives everyone who left.

In [ ]:
# Exercise 2.2
# Task: compare the patients still in the panel with those who left (after year 5):
#   n, mean Glucose, Outcome rate. One row each.
# Expected output: leavers have higher mean glucose and a higher outcome rate.
# YOUR CODE HERE

Question: a risk model is trained on the year-5 panel. Are its predicted diabetes rates for new patients too high or too low?

Fix: keep departed rows with an exit date and a reason. Report the original cohort, not the survivors.

Takeaway: a filter related to the outcome can manufacture a trend.

## §3 Sampling distributions

Problem: two analysts each draw 50 patients at random from the same population. Their mean glucose values differ by several mg/dL.
Who made a mistake? Neither.

Idea: an estimate is one draw from a distribution of possible estimates.

- Sampling distribution: the distribution of an estimate over repeated samples.
- Standard error (SE): the standard deviation of that distribution.
- For a mean: SE = σ / √n.

The 763 patients of §1 still play the population. Real work has one sample, which is the problem of Session B.

Note: the draws below are with replacement, so every draw comes from the same distribution. Surveys draw without replacement;
when n is far below the population size the difference is small.

In [ ]:
rng = np.random.default_rng(5110)
values = pop["Glucose"].to_numpy()
sigma = values.std()          # population standard deviation

def sample_means(source, n, reps=2000):
    return rng.choice(source, size=(reps, n)).mean(axis=1)

round(sigma, 1)

Expected: about 30.5 mg/dL.

In [ ]:
# Exercise 3.1
# Task: draw 2000 samples of n = 50 with sample_means(values, 50). Plot a histogram of the means
#   with a vertical line at mu. Print the standard deviation of the means and sigma / sqrt(50).
# Expected output: a bell-shaped histogram centred on mu. Both numbers about 4.3.
# YOUR CODE HERE

In [ ]:
# Exercise 3.2
# Task: repeat for n = 25, 100, 400. Build a table with columns n, simulated_se, formula_se.
# Expected output: about 6.1, 3.0, 1.5 in both columns. Four times the data halves the SE.
# YOUR CODE HERE

In [ ]:
# Exercise 3.3
# Task: Insulin is skewed. Plot two histograms side by side:
#   (a) the observed Insulin values, (b) 2000 means of samples of n = 50 drawn from them.
# Expected output: (a) long right tail. (b) close to symmetric, centred near 155.
# YOUR CODE HERE

In other words: the central limit theorem at work. Means of independent draws look normal even when the data do not.
Medians, maxima and very small samples get no such guarantee.

Note: SE shrinks with n. Bias does not. The volunteer study of §1 with 10 times more volunteers is just as wrong, only more precise.

Takeaway: report the standard error with every estimate. It shrinks like 1/√n.

## §4 Bootstrap and confidence intervals

Problem: the 768 patients are the only sample. §3 needed a population to draw from. Real work never has one.
Mean glucose is 121.7 mg/dL. Plus or minus what?

Idea: resample the sample (Efron, 1979).

1. Draw n rows with replacement from the data.
2. Compute the statistic.
3. Repeat B times.

- Bootstrap SE: the standard deviation of the B statistics.
- 95% percentile interval: their 2.5th and 97.5th percentiles.

Note: the same code works for a median, a ratio or a correlation. No formula needed.

In [ ]:
rng = np.random.default_rng(5110)

def bootstrap(data, stat=np.mean, B=5000):
    data = np.asarray(data)
    idx = rng.integers(0, len(data), size=(B, len(data)))   # B resamples, each of size n
    return stat(data[idx], axis=1)

In [ ]:
# Exercise 4.1
# Task: bootstrap the mean of the observed Glucose values. Print the bootstrap SE
#   and the 95% percentile interval (np.percentile).
# Expected output: SE about 1.1. Interval about [119.5, 123.9].
# YOUR CODE HERE

Question: what does "95%" refer to? The interval, the true mean, or the procedure?

Check it by simulation. Back to the §1 device: pop plays the population, so mu is known.

In [ ]:
# Exercise 4.2
# Task: 500 times, draw a sample of n = 50 from values (rng.choice), compute its 95% bootstrap
#   percentile interval with B = 2000, and record whether the interval contains mu.
#   Report the share of intervals that cover mu.
# Expected output: close to 0.95, often slightly below. Percentile intervals run a little narrow at n = 50.
# YOUR CODE HERE

In other words: 95% is a property of the procedure. A single interval either covers the true value or does not.

Next: Week 8 §5 said single imputation makes intervals too narrow. Measure it.

In [ ]:
# Exercise 4.3
# Task: 95% bootstrap interval for mean Insulin, twice:
#   (a) the 394 observed values, (b) all 768 values after median fill (fillna with the observed median).
#   Table with rows observed and median_filled; columns n, mean, lo, hi, width.
# Expected output: observed: mean about 155, width about 23.
#   Median-filled: mean about 141, width about 12. Half as wide, and centred lower.
# YOUR CODE HERE

Question: which interval belongs in a report? What false claim does the other one make?

Note: resample the unit that was sampled. Repeated visits per patient → resample patients, not visits.
Time series → blocks of time. Week 12 applies the same rule to cross-validation.

Takeaway: report an interval with every estimate, and say which errors it leaves out. It covers sampling noise only:
not selection, not survivorship, not a bad fill.

## §5 Permutation tests and p-values

Problem: Week 8 §4.1 compared patients with and without an Insulin value and saw differences.
Any split of 768 rows into two groups gives some difference. Is this one bigger than chance would give?

Idea: if the group label has nothing to do with the value, shuffling the labels changes nothing systematic.
Shuffle many times and see how often a difference this large appears.

- Null hypothesis H0: the label is unrelated to the value.
- Test statistic: difference in group means.
- p-value: probability, if H0 is true, of a statistic at least as extreme as the one observed.

In [ ]:
rng = np.random.default_rng(5110)

def perm_test(data, labels, reps=5000):
    data, labels = np.asarray(data, dtype=float), np.asarray(labels, dtype=bool)
    def diff(lab):
        return data[lab].mean() - data[~lab].mean()
    observed = diff(labels)
    null = np.array([diff(rng.permutation(labels)) for _ in range(reps)])
    p = (np.sum(np.abs(null) >= abs(observed)) + 1) / (reps + 1)   # two-sided
    return observed, p, null

Note: `+ 1` counts the observed arrangement as one of the possible shuffles. Smallest reportable p: 1/5001.

Note: the Welch t-test (`scipy.stats.ttest_ind(..., equal_var=False)`) answers the same question with a formula. The shuffle makes the logic visible.

In [ ]:
# Exercise 5.1
# Task: test mean Glucose by Outcome (Outcome == 1 vs 0), on rows where Glucose is observed.
#   Print the observed difference and p. Plot a histogram of null with a vertical line at the observed difference.
# Expected output: difference about 32 mg/dL. The null histogram sits within about ±7.
#   No shuffle reaches the observed value → p = 1/5001, report as p < 0.001.
# YOUR CODE HERE

In [ ]:
# Exercise 5.2 (Week 8 §4.1, tested)
# Task: test the Outcome rate by Insulin missingness: labels df["Insulin"].isna(), values df["Outcome"].
# Expected output: a difference of a few percentage points. p well above 0.05.
# YOUR CODE HERE

Question: 5.2 gives a large p-value. Is Insulin missing completely at random?

In other words: a large p-value means no evidence against H0 from this statistic. It is not evidence for H0.

In [ ]:
# Exercise 5.3
# Task: test mean BloodPressure by Outcome, on rows where BloodPressure is observed.
# Expected output: difference about 4 mm Hg, p < 0.001.
#   Then answer in a comment: is a 4 mm Hg difference in diastolic pressure large?
# YOUR CODE HERE

Note: what a p-value is not (Wasserstein and Lazar, 2016).
- Not the probability that H0 is true.
- Not the size or importance of an effect.
- p > 0.05 is not evidence of no effect.

Takeaway: a p-value measures surprise under H0. Report the effect size and its interval first.

## §6 Multiple comparisons

Problem: an analyst tests every column against Outcome and reports the ones with p < 0.05.
Here the table gets 20 extra columns of pure random noise. Some of them will "pass".

Idea: count the tests. Control the error rate across all of them, not per test.

- 20 tests, every H0 true, threshold 0.05: about 1 false alarm expected.
- Probability of at least one: 1 - 0.95^20 ≈ 0.64.

Note: 28 shuffle tests of 5000 permutations each are slow. This section uses the Welch t-test.

In [ ]:
from scipy import stats

noise_rng = np.random.default_rng(2016)
noise = pd.DataFrame(noise_rng.normal(size=(len(df), 20)),
                     columns=[f"noise_{i:02d}" for i in range(1, 21)])
wide = pd.concat([df, noise], axis=1)
TEST_COLS = FEATURES + list(noise.columns)
y = df["Outcome"]

def welch_p(s):
    return stats.ttest_ind(s[y == 1].dropna(), s[y == 0].dropna(), equal_var=False).pvalue

len(TEST_COLS)

In [ ]:
# Exercise 6.1
# Task: compute welch_p for every column in TEST_COLS (a Series called pvals).
#   Count p < 0.05 separately for the 8 real columns and the 20 noise columns.
# Expected output: all 8 real columns below 0.05. Noise: about 1 (0 to 3 is common).
# YOUR CODE HERE

In [ ]:
# Exercise 6.2
# Task: 1000 rounds. In each, draw a fresh (768, 20) noise matrix with noise_rng.normal,
#   t-test all 20 columns at once (stats.ttest_ind(Z[y1], Z[~y1], axis=0, equal_var=False)),
#   and record whether any p < 0.05. Report the share of rounds with at least one false alarm.
# Expected output: about 0.64.
# YOUR CODE HERE

Two corrections:
- Bonferroni: reject only when p < 0.05 / m. Controls the chance of any false alarm. Strict when m is large.
- Benjamini and Hochberg (1995): sort the p-values; find the largest i with p(i) ≤ 0.05 · i / m; reject the i smallest.
  Controls the false discovery rate, the expected share of false alarms among the rejections.

Note: `statsmodels.stats.multitest.multipletests(p, method="bonferroni")` and `method="fdr_bh"` give the same answers as the code below.

In [ ]:
def bh_reject(p, alpha=0.05):
    p = np.asarray(p)
    m = len(p)
    order = np.argsort(p)
    passed = p[order] <= alpha * np.arange(1, m + 1) / m
    k = passed.nonzero()[0].max() + 1 if passed.any() else 0
    reject = np.zeros(m, dtype=bool)
    reject[order[:k]] = True
    return reject

In [ ]:
# Exercise 6.3
# Task: apply Bonferroni (pvals < 0.05 / len(pvals)) and bh_reject(pvals) to the 6.1 p-values.
#   Table: rows none, bonferroni, bh; columns real, noise; entries = number of rejections.
# Expected output: noise falls to 0 under both corrections (almost always).
#   All or nearly all real columns survive.
# YOUR CODE HERE

Note: tests also hide in choices. Four imputation methods tried, one reported (Week 8). Three outlier rules, one reported.
Subgroups picked after seeing the data. Each path tried is a test (Gelman and Loken, 2013).

Takeaway: a p-value means little without the number of tests behind it. Report them all, or correct.

## Link to homework and project

Project prototype review (due Week 11): every preliminary finding states its target population and selection rule (§1, §2),
carries a bootstrap interval (§4.1), and says how many comparisons were tried (§6).

Homework 6 (due Week 13): the required account of "what you cannot conclude" grows out of §1 to §4.
Name the population the data represent, the filters that removed rows, and the interval around each reported number.

Homework 5 (due Week 10): §4.3 is one more piece of evidence for an imputation defence.
Show how the chosen fill changes the interval of a key statistic.